## 🔬 Análise Visual das Notas dos Alunos (`student_scores.json`)

Este notebook carrega o arquivo `student_scores.json` com a estrutura de notas aninhada por trimestre (T1, T2, T3), processa os dados para um formato tabular (DataFrame) e gera visualizações interativas para analisar a performance dos alunos.

### 1. Importação das Bibliotecas

In [ ]:
import json
import pandas as pd
import plotly.express as px

# Configuração para exibir todas as colunas do DataFrame
pd.set_option('display.max_columns', None)

### 2. Carregar e Processar os Dados

O arquivo JSON possui uma estrutura aninhada por trimestres. Vamos achatá-la para criar um DataFrame onde cada linha representa o conjunto de notas de um aluno em uma disciplina. Em seguida, calcularemos as médias de cada trimestre.

In [ ]:
# Altere o caminho se o notebook não estiver na mesma pasta que a pasta 'data'
file_path = 'data/repo/plugins/student_scores.json'

with open(file_path, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

students_data = raw_data.get('students_data', {})

# Lista para armazenar os dados processados
processed_data = []

# Iterar sobre cada aluno e suas disciplinas
for student_id, student_info in students_data.items():
    student_name = student_info.get('name')
    qualitative_points = sum(p.get('points', 0) for p in student_info.get('daily_qualitative_points', []))

    # Iterar sobre as disciplinas do aluno
    for subject_id, subject_details in student_info.get('subjects', {}).items():
        row = {
            'ID Aluno': student_id,
            'Nome': student_name,
            'ID Disciplina': subject_id,
            'Total Pontos Qualitativos (Geral)': qualitative_points
        }
        
        # Extrair notas de cada trimestre da estrutura aninhada
        for t in ['T1', 'T2', 'T3']:
            trim_data = subject_details.get(t, {})
            for n in ['N1', 'N2', 'N3']:
                row[f'{t}_{n}'] = trim_data.get(n, 0.0)
        
        processed_data.append(row)

# Criar o DataFrame
df = pd.DataFrame(processed_data)

# Função para calcular a média de um conjunto de notas, ignorando zeros
def calculate_mean_ignoring_zeros(row, cols):
    notes = [row[c] for c in cols if row[c] > 0]
    return sum(notes) / len(notes) if notes else 0.0

# Calcular as médias trimestrais
df['Média T1'] = df.apply(lambda row: calculate_mean_ignoring_zeros(row, ['T1_N1', 'T1_N2', 'T1_N3']), axis=1)
df['Média T2'] = df.apply(lambda row: calculate_mean_ignoring_zeros(row, ['T2_N1', 'T2_N2', 'T2_N3']), axis=1)
df['Média T3'] = df.apply(lambda row: calculate_mean_ignoring_zeros(row, ['T3_N1', 'T3_N2', 'T3_N3']), axis=1)

# Exibir as primeiras linhas para verificação
df.head()

### 3. Análise Descritiva

Vamos ver um resumo estatístico das médias trimestrais.

In [ ]:
df[['Média T1', 'Média T2', 'Média T3']].describe()

### 4. Visualizações Interativas

#### Gráfico 1: Média Geral por Trimestre em Cada Disciplina
Este gráfico nos ajuda a comparar a performance média das turmas em cada trimestre por disciplina.

In [ ]:
# Agrupar por disciplina e calcular a média
avg_scores_by_subject = df.groupby('ID Disciplina')[['Média T1', 'Média T2', 'Média T3']].mean().reset_index()

# Reformatar os dados para o Plotly (formato longo)
avg_scores_melted = avg_scores_by_subject.melt(
    id_vars='ID Disciplina', 
    value_vars=['Média T1', 'Média T2', 'Média T3'], 
    var_name='Trimestre', 
    value_name='Média'
)

# Criar o gráfico de barras
fig1 = px.bar(
    avg_scores_melted, 
    x='ID Disciplina', 
    y='Média', 
    color='Trimestre', 
    barmode='group', # Agrupa as barras lado a lado
    title='Média Geral por Trimestre em Cada Disciplina',
    labels={'ID Disciplina': 'ID da Disciplina', 'Média': 'Nota Média'},
    text_auto='.2f' # Exibe o valor em cima da barra
)

fig1.show()

#### Gráfico 2: Distribuição de Pontos Qualitativos por Aluno

Este gráfico mostra quais alunos mais acumularam pontos por participação e atividades diárias.

In [ ]:
# Pegar os pontos gerais por aluno (evitando duplicatas)
qualitative_by_student = df[['Nome', 'Total Pontos Qualitativos (Geral)']].drop_duplicates().sort_values(by='Total Pontos Qualitativos (Geral)', ascending=False)

fig2 = px.bar(
    qualitative_by_student.head(20), # Mostra os 20 melhores
    x='Nome',
    y='Total Pontos Qualitativos (Geral)',
    title='Top 20 Alunos por Pontos Qualitativos Acumulados',
    labels={'Nome': 'Aluno', 'Total Pontos Qualitativos (Geral)': 'Total de Pontos'},
    text_auto='.2f'
)

fig2.update_layout(xaxis_tickangle=-45)
fig2.show()

### 5. Análise Individual de Aluno

Vamos focar em um aluno específico para ver sua performance trimestral em cada disciplina.

In [ ]:
# Escolha o aluno pelo nome
student_name_to_analyze = "LUARA CRISTINE VISGUEIRA DE SOUSA LEMOS"

# Filtrar o DataFrame para o aluno específico
student_df = df[df['Nome'] == student_name_to_analyze]

# Reformatar os dados do aluno para o formato longo
student_melted = student_df.melt(
    id_vars=['Nome', 'ID Disciplina'],
    value_vars=['Média T1', 'Média T2', 'Média T3'],
    var_name='Trimestre',
    value_name='Nota'
)

# Criar o gráfico de barras para o aluno
fig3 = px.bar(
    student_melted,
    x='ID Disciplina',
    y='Nota',
    color='Trimestre',
    barmode='group',
    title=f'Desempenho Trimestral de {student_name_to_analyze} por Disciplina',
    labels={'ID Disciplina': 'ID da Disciplina', 'Nota': 'Média do Trimestre'},
    text_auto='.2f'
)

fig3.show()